### Dataset used is DATA2017 - MF 

In [1]:
# Data manipulation and analysis
import numpy as np
import pandas as pd

# Data preprocessing and machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing import sequence

# TensorFlow and Keras for building the model
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses
from tensorflow.keras.callbacks import EarlyStopping

# Metrics for evaluation
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score

# Visualization (if needed)
import matplotlib.pyplot as plt
import seaborn as sns




In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the train and test datasets
train_file_path = '/kaggle/input/data2017-dataset/Data2017/mf/trainData.csv'  # Replace with your train dataset file path
test_file_path = '/kaggle/input/data2017-dataset/Data2017/mf/testData.csv'    # Replace with your test dataset file path

train_data = pd.read_csv(train_file_path, header=None)
test_data = pd.read_csv(test_file_path, header=None)

# Separate features (protein sequences) and labels for both train and test datasets
x_train = train_data.iloc[:, 0].values  # Protein sequences in the first column of the train dataset
y_train = train_data.iloc[:, 1:].values  # Labels in the rest of the columns of the train dataset

x_test = test_data.iloc[:, 0].values  # Protein sequences in the first column of the test dataset
y_test = test_data.iloc[:, 1:].values  # Labels in the rest of the columns of the test dataset



In [3]:
import numpy as np

# Check the datatype of the sequences
print(f"Datatype of sequences in x_train: {type(x_train)}")
print(f"Datatype of sequences in y_train: {type(y_train)}")
print(f"Datatype of sequences in x_test: {type(x_test)}")
print(f"Datatype of sequences in y_test: {type(y_test)}")

# Check the number of rows in train and test datasets
print(f"Number of rows in train dataset: {x_train.shape[0]}")
print(f"Number of rows in test dataset: {x_test.shape[0]}")

# Check length of each sequence in the train dataset
sequence_lengths_train = [len(seq) for seq in x_train]

# Calculate the maximum, minimum, average, and median length of sequences in the train dataset
max_length_train = np.max(sequence_lengths_train)
min_length_train = np.min(sequence_lengths_train)
avg_length_train = np.mean(sequence_lengths_train)
median_length_train = np.median(sequence_lengths_train)

# Print the results
print(f"Maximum length of sequences in train dataset: {max_length_train}")
print(f"Minimum length of sequences in train dataset: {min_length_train}")
print(f"Average length of sequences in train dataset: {avg_length_train}")
print(f"Median length of sequences in train dataset: {median_length_train}")


Datatype of sequences in x_train: <class 'numpy.ndarray'>
Datatype of sequences in y_train: <class 'numpy.ndarray'>
Datatype of sequences in x_test: <class 'numpy.ndarray'>
Datatype of sequences in y_test: <class 'numpy.ndarray'>
Number of rows in train dataset: 32280
Number of rows in test dataset: 3132
Maximum length of sequences in train dataset: 1998
Minimum length of sequences in train dataset: 61
Average length of sequences in train dataset: 496.11806071871126
Median length of sequences in train dataset: 415.0


In [4]:
import numpy as np

# Get the length of each sequence in the train dataset
train_lengths = [len(sequence) for sequence in x_train]

# Define the length ranges (as provided)
length_ranges = [
    (0, 100),
    (101, 200),
    (201, 300),
    (301, 400),
    (401, 500),
    (501, 600),
    (601, 700),
    (701, 800),
    (801, 900),
    (901, 1000),
    (1001, 1100),
    (1101, 1200),
    (1201, 1300),
    (1301, 1400),
    (1401, 1500),
    (1501, 1600),
    (1601, 1700),
    (1701, 1800),
    (1801, 1900),
    (1901, 2000)
]

# Count the number of sequences in each length range
length_counts = {}

for lower, upper in length_ranges:
    count = sum((np.array(train_lengths) >= lower) & (np.array(train_lengths) <= upper))
    length_counts[f"{lower}-{upper}"] = count

# Print the counts for each range
for length_range, count in length_counts.items():
    print(f"Sequences with length {length_range}: {count}")


Sequences with length 0-100: 607
Sequences with length 101-200: 4294
Sequences with length 201-300: 4650
Sequences with length 301-400: 5861
Sequences with length 401-500: 4904
Sequences with length 501-600: 3390
Sequences with length 601-700: 2159
Sequences with length 701-800: 1595
Sequences with length 801-900: 1199
Sequences with length 901-1000: 871
Sequences with length 1001-1100: 714
Sequences with length 1101-1200: 491
Sequences with length 1201-1300: 359
Sequences with length 1301-1400: 302
Sequences with length 1401-1500: 272
Sequences with length 1501-1600: 158
Sequences with length 1601-1700: 111
Sequences with length 1701-1800: 139
Sequences with length 1801-1900: 101
Sequences with length 1901-2000: 103


### Number of Sequences of Sequence length 0-300 = 9551, 301-500 = 10,765 , 501-900 = 8343, 901-2000 = 3621
### Training set is now divided to 4 parts : I. Seq 0-300 length
### II. Seq 301-500 , III. Seq 501-900, IV. Seq length 901-2000

In [5]:
import numpy as np

# Define the length ranges
length_ranges = [
    (0, 300),  # Seq 0-300 length
    (301, 500),  # Seq 301-500 length
    (501, 900),  # Seq 501-900 length
    (901, 2000)  # Seq 901-2000 length
]

# Get the length of each sequence in the train dataset
train_lengths = [len(sequence) for sequence in x_train]

# Initialize lists to store the training sets for each length range
x_train_1 = []
y_train_1 = []

x_train_2 = []
y_train_2 = []

x_train_3 = []
y_train_3 = []

x_train_4 = []
y_train_4 = []

# Divide the train dataset based on length ranges
for i, length in enumerate(train_lengths):
    if 0 <= length <= 300:
        x_train_1.append(x_train[i])
        y_train_1.append(y_train[i])
    elif 301 <= length <= 500:
        x_train_2.append(x_train[i])
        y_train_2.append(y_train[i])
    elif 501 <= length <= 900:
        x_train_3.append(x_train[i])
        y_train_3.append(y_train[i])
    elif 901 <= length <= 2000:
        x_train_4.append(x_train[i])
        y_train_4.append(y_train[i])

# Convert lists to numpy arrays or pandas DataFrames as needed
x_train_1 = np.array(x_train_1)
y_train_1 = np.array(y_train_1)

x_train_2 = np.array(x_train_2)
y_train_2 = np.array(y_train_2)

x_train_3 = np.array(x_train_3)
y_train_3 = np.array(y_train_3)

x_train_4 = np.array(x_train_4)
y_train_4 = np.array(y_train_4)

# Now you have four separate datasets based on sequence length ranges
print(f"Training set 1 (Seq 0-300 length): {len(x_train_1)} sequences")
print(f"Training set 2 (Seq 301-500 length): {len(x_train_2)} sequences")
print(f"Training set 3 (Seq 501-900 length): {len(x_train_3)} sequences")
print(f"Training set 4 (Seq 901-2000 length): {len(x_train_4)} sequences")


Training set 1 (Seq 0-300 length): 9551 sequences
Training set 2 (Seq 301-500 length): 10765 sequences
Training set 3 (Seq 501-900 length): 8343 sequences
Training set 4 (Seq 901-2000 length): 3621 sequences


In [6]:
import numpy as np
import math

# Function for segmentation
def segment(dataset, labels, seg_size, overlap):
    seg_data, seg_labels = [], []
    for j, row in enumerate(dataset):
        # Ensure each row is treated as a sequence
        if isinstance(row, (np.ndarray, list, str)) and len(row) >= seg_size:
            pos = math.ceil(len(row) / overlap)
            if pos < math.ceil(seg_size / overlap):
                pos = math.ceil(seg_size / overlap)
            for itr in range(pos - math.ceil(seg_size / overlap) + 1):
                init = itr * overlap
                segment = row[init:init + seg_size]
                if len(segment) == seg_size:  # Ensure full segments
                    seg_data.append(segment)
                    seg_labels.append(labels[j])
    return np.array(seg_data), np.array(seg_labels)

# Function to create the n-gram dictionary
def create_ngram_dictionary(dataset, chunk_size):
    ngram_dict = {}
    idx = 0
    for row in dataset:
        for i in range(len(row) - chunk_size + 1):
            key = tuple(row[i:i + chunk_size])
            if key not in ngram_dict:
                ngram_dict[key] = idx
                idx += 1
    return ngram_dict

# Function to encode sequences into n-grams
def ngram_encode(dataset, chunk_size, ngram_dict):
    encoded_data = []
    for row in dataset:
        encoded_seq = []
        for i in range(len(row) - chunk_size + 1):
            key = tuple(row[i:i + chunk_size])
            if key in ngram_dict:
                encoded_seq.append(ngram_dict[key])
        encoded_data.append(encoded_seq)
    return np.array(encoded_data)

# Parameters
seg_size = 200
overlap = 50
chunk_size = 4

# Step 1: Segmentation
x_train_segments, y_train_segments = {}, {}
for i, (x_train_part, y_train_part) in enumerate(zip(
        [x_train_1, x_train_2, x_train_3, x_train_4],
        [y_train_1, y_train_2, y_train_3, y_train_4]
    ), 1):
    x_train_segments[f"x_train_{i}_seg"], y_train_segments[f"y_train_{i}_seg"] = segment(
        x_train_part, y_train_part, seg_size, overlap
    )

x_test_seg, y_test_seg = segment(x_test, y_test, seg_size, overlap)

# Step 2: n-Gram Dictionary Creation
combined_sequences = []
for i in range(1, 5):
    combined_sequences.extend(x_train_segments[f"x_train_{i}_seg"])
combined_sequences.extend(x_test_seg)

ngram_dict = create_ngram_dictionary(combined_sequences, chunk_size)

# Step 3: n-Gram Encoding
x_train_ngram, y_train_ngram = {}, {}
for i in range(1, 5):
    x_train_ngram[f"x_train_{i}_ngram"] = ngram_encode(
        x_train_segments[f"x_train_{i}_seg"], chunk_size, ngram_dict
    )
    y_train_ngram[f"y_train_{i}_ngram"] = y_train_segments[f"y_train_{i}_seg"]

x_test_ngram = ngram_encode(x_test_seg, chunk_size, ngram_dict)
y_test_ngram = y_test_seg

# Summary
print("\nPreprocessing Summary:")
for i in range(1, 5):
    print(f"Training set {i}: {len(x_train_ngram[f'x_train_{i}_ngram'])} sequences.")
print(f"Test set: {len(x_test_ngram)} sequences.")
print(f"n-Gram dictionary size: {len(ngram_dict)}")



Preprocessing Summary:
Training set 1: 7042 sequences.
Training set 2: 47419 sequences.
Training set 3: 80181 sequences.
Training set 4: 76321 sequences.
Test set: 0 sequences.
n-Gram dictionary size: 159750


In [24]:
from tensorflow.keras import layers, models, optimizers, losses
import tensorflow as tf

def DC_CNN_Block(nb_filter, filter_length, dilation, l2_layer_reg):
    def f(input_):
        residual = input_
        layer_out = layers.Conv1D(filters=nb_filter, kernel_size=filter_length, dilation_rate=dilation,
                                  activation='linear', padding='same', use_bias=True)(input_)
        layer_out = layers.BatchNormalization(epsilon=1.1e-5)(layer_out)
        layer_out = layers.LeakyReLU(alpha=0.2)(layer_out)

        # Compute attention for each step
        attention1 = layers.Dense(1, activation='tanh')(layer_out)
        attention1 = layers.Flatten()(attention1)
        attention1 = layers.Activation('softmax')(attention1)

        attention2 = layers.Dense(1, activation='tanh')(layer_out)
        attention2 = layers.Flatten()(attention2)
        attention2 = layers.Activation('softmax')(attention2)

        attention3 = layers.Dense(1, activation='tanh')(layer_out)
        attention3 = layers.Flatten()(attention3)
        attention3 = layers.Activation('softmax')(attention3)

        attention = layers.Add()([attention1, attention2, attention3])
        attention = layers.RepeatVector(nb_filter)(attention)
        attention = layers.Permute([2, 1])(attention)

        sent_representation = layers.Multiply()([layer_out, attention])
        sent_representation = layers.Lambda(lambda xin: tf.reduce_sum(xin, axis=1))(sent_representation)
        return sent_representation
    return f

embed_dim = 64
ff_dim = 960

def DC_CNN_Model(top_words, seq_len, o_dim):
    f_num = 192
    f_size = [6, 6, 6, 6, 6]

    _input = layers.Input(shape=(seq_len,))
    emd = layers.Embedding(top_words, embed_dim, input_length=seq_len)(_input)
    drop1 = layers.Dropout(0.3)(emd)

    l1 = DC_CNN_Block(f_num, f_size[0], 1, 0.001)(drop1)
    l2 = DC_CNN_Block(f_num, f_size[1], 3, 0.001)(drop1)
    l3 = DC_CNN_Block(f_num, f_size[2], 5, 0.001)(drop1)
    l4 = DC_CNN_Block(f_num, f_size[3], 7, 0.001)(drop1)
    l5 = DC_CNN_Block(f_num, f_size[4], 9, 0.001)(drop1)

    x = layers.Concatenate(axis=-1)([l1, l2, l3, l4, l5])

    x = layers.Dropout(0.4)(x)
    _output = layers.Dense(o_dim, kernel_initializer='normal', activation='softmax', name='CLASSIFIER')(x)  # Softmax for multi-class classification

    model = models.Model(inputs=_input, outputs=_output)
    model.compile(
        loss=losses.CategoricalCrossentropy(),  # Use categorical crossentropy for multi-class
        optimizer=optimizers.Adam(learning_rate=0.0005),
        metrics=[tf.keras.metrics.CategoricalAccuracy()]
    )
    return model


In [25]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# Function to pad the sequences
def pad_data(sequences, seq_len):
    return pad_sequences(sequences, maxlen=seq_len, padding='post', truncating='post')

# Padding the training data
x_train_padded = {}
for i in range(1, 5):
    x_train_padded[f"x_train_{i}_padded"] = pad_data(x_train_ngram[f"x_train_{i}_ngram"], seg_size)

# Padding the test data
x_test_padded = pad_data(x_test_ngram, seg_size)

# One-hot encoding the labels (multiclass)
y_train_padded = {}
for i in range(1, 5):
    y_train_padded[f"y_train_{i}_padded"] = to_categorical(y_train_ngram[f"y_train_{i}_ngram"], num_classes=o_dim)

y_test_padded = to_categorical(y_test_ngram, num_classes=o_dim)


In [27]:
'''from tensorflow.keras.callbacks import ModelCheckpoint
import os

top_words = len(ngram_dict)  # This will be the size of the n-gram dictionary.
o_dim = len(np.unique(y_train))  # For multiclass, set o_dim to the number of unique classes in y_train

# Function to train and save the model incrementally
def train_and_save_model(x_train_data, y_train_data, model_name, model):
    checkpoint = ModelCheckpoint(model_name, monitor='loss', verbose=1, save_best_only=True)
    model.fit(x_train_data, y_train_data, epochs=10, batch_size=32, validation_split=0.2, callbacks=[checkpoint])

# Directory to save models
model_dir = "saved_models"
os.makedirs(model_dir, exist_ok=True)

# Phase 1: Train on x_train_1_ngram
model1 = DC_CNN_Model(top_words, seg_size, o_dim)  # Initialize the model
train_and_save_model(x_train_padded["x_train_1_padded"], y_train_padded["y_train_1_padded"], 
                     os.path.join(model_dir, "model1.keras"), model1)

# Phase 2: Train on x_train_1_ngram and x_train_2_ngram
model2 = DC_CNN_Model(top_words, seg_size, o_dim)  # Re-initialize the model
train_and_save_model(x_train_padded["x_train_1_padded"], y_train_padded["y_train_1_padded"], 
                     os.path.join(model_dir, "model2.keras"), model2)
train_and_save_model(x_train_padded["x_train_2_padded"], y_train_padded["y_train_2_padded"], 
                     os.path.join(model_dir, "model2.keras"), model2)

# Phase 3: Train on x_train_1_ngram, x_train_2_ngram, and x_train_3_ngram
model3 = DC_CNN_Model(top_words, seg_size, o_dim)  # Re-initialize the model
train_and_save_model(x_train_padded["x_train_1_padded"], y_train_padded["y_train_1_padded"], 
                     os.path.join(model_dir, "model3.keras"), model3)
train_and_save_model(x_train_padded["x_train_2_padded"], y_train_padded["y_train_2_padded"], 
                     os.path.join(model_dir, "model3.keras"), model3)
train_and_save_model(x_train_padded["x_train_3_padded"], y_train_padded["y_train_3_padded"], 
                     os.path.join(model_dir, "model3.keras"), model3)

# Phase 4: Train on x_train_1_ngram, x_train_2_ngram, x_train_3_ngram, and x_train_4_ngram
model4 = DC_CNN_Model(top_words, seg_size, o_dim)  # Re-initialize the model
train_and_save_model(x_train_padded["x_train_1_padded"], y_train_padded["y_train_1_padded"], 
                     os.path.join(model_dir, "model4.keras"), model4)
train_and_save_model(x_train_padded["x_train_2_padded"], y_train_padded["y_train_2_padded"], 
                     os.path.join(model_dir, "model4.keras"), model4)
train_and_save_model(x_train_padded["x_train_3_padded"], y_train_padded["y_train_3_padded"], 
                     os.path.join(model_dir, "model4.keras"), model4)
train_and_save_model(x_train_padded["x_train_4_padded"], y_train_padded["y_train_4_padded"], 
                     os.path.join(model_dir, "model4.keras"), model4)
'''

'from tensorflow.keras.callbacks import ModelCheckpoint\nimport os\n\ntop_words = len(ngram_dict)  # This will be the size of the n-gram dictionary.\no_dim = len(np.unique(y_train))  # For multiclass, set o_dim to the number of unique classes in y_train\n\n# Function to train and save the model incrementally\ndef train_and_save_model(x_train_data, y_train_data, model_name, model):\n    checkpoint = ModelCheckpoint(model_name, monitor=\'loss\', verbose=1, save_best_only=True)\n    model.fit(x_train_data, y_train_data, epochs=10, batch_size=32, validation_split=0.2, callbacks=[checkpoint])\n\n# Directory to save models\nmodel_dir = "saved_models"\nos.makedirs(model_dir, exist_ok=True)\n\n# Phase 1: Train on x_train_1_ngram\nmodel1 = DC_CNN_Model(top_words, seg_size, o_dim)  # Initialize the model\ntrain_and_save_model(x_train_padded["x_train_1_padded"], y_train_padded["y_train_1_padded"], \n                     os.path.join(model_dir, "model1.keras"), model1)\n\n# Phase 2: Train on x_tra

In [29]:
from tensorflow.keras.callbacks import ModelCheckpoint
import os

top_words = len(ngram_dict)  # Size of the n-gram dictionary
o_dim = len(np.unique(y_train))  # Number of unique classes in y_train

# Function to train and save the model incrementally
def train_and_save_model(x_train_data, y_train_data, model_name, model):
    checkpoint = ModelCheckpoint(model_name, monitor='loss', verbose=1, save_best_only=True)
    model.fit(x_train_data, y_train_data, epochs=10, batch_size=32, validation_split=0.2, callbacks=[checkpoint])

# Directory to save models
model_dir = "saved_models"
os.makedirs(model_dir, exist_ok=True)

# Ensure your labels are flattened (if needed)
y_train_padded_flattened = {key: np.reshape(value, (-1, o_dim)) for key, value in y_train_padded.items()}
y_test_padded_flattened = np.reshape(y_test_padded, (-1, o_dim))

# Phase 1: Train on x_train_1_ngram
model1 = DC_CNN_Model(top_words, seg_size, o_dim)  # Initialize the model
train_and_save_model(x_train_padded["x_train_1_padded"], y_train_padded_flattened["y_train_1_padded"], 
                     os.path.join(model_dir, "model1.keras"), model1)

# Phase 2: Train on x_train_1_ngram and x_train_2_ngram
model2 = DC_CNN_Model(top_words, seg_size, o_dim)  # Re-initialize the model
train_and_save_model(x_train_padded["x_train_1_padded"], y_train_padded_flattened["y_train_1_padded"], 
                     os.path.join(model_dir, "model2.keras"), model2)
train_and_save_model(x_train_padded["x_train_2_padded"], y_train_padded_flattened["y_train_2_padded"], 
                     os.path.join(model_dir, "model2.keras"), model2)

# Phase 3: Train on x_train_1_ngram, x_train_2_ngram, and x_train_3_ngram
model3 = DC_CNN_Model(top_words, seg_size, o_dim)  # Re-initialize the model
train_and_save_model(x_train_padded["x_train_1_padded"], y_train_padded_flattened["y_train_1_padded"], 
                     os.path.join(model_dir, "model3.keras"), model3)
train_and_save_model(x_train_padded["x_train_2_padded"], y_train_padded_flattened["y_train_2_padded"], 
                     os.path.join(model_dir, "model3.keras"), model3)
train_and_save_model(x_train_padded["x_train_3_padded"], y_train_padded_flattened["y_train_3_padded"], 
                     os.path.join(model_dir, "model3.keras"), model3)

# Phase 4: Train on x_train_1_ngram, x_train_2_ngram, x_train_3_ngram, and x_train_4_ngram
model4 = DC_CNN_Model(top_words, seg_size, o_dim)  # Re-initialize the model
train_and_save_model(x_train_padded["x_train_1_padded"], y_train_padded_flattened["y_train_1_padded"], 
                     os.path.join(model_dir, "model4.keras"), model4)
train_and_save_model(x_train_padded["x_train_2_padded"], y_train_padded_flattened["y_train_2_padded"], 
                     os.path.join(model_dir, "model4.keras"), model4)
train_and_save_model(x_train_padded["x_train_3_padded"], y_train_padded_flattened["y_train_3_padded"], 
                     os.path.join(model_dir, "model4.keras"), model4)
train_and_save_model(x_train_padded["x_train_4_padded"], y_train_padded_flattened["y_train_4_padded"], 
                     os.path.join(model_dir, "model4.keras"), model4)


Epoch 1/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - categorical_accuracy: 0.9424 - loss: 0.1881
Epoch 1: loss improved from inf to 0.11005, saving model to saved_models/model1.keras
177/177 ━━━━━━━━━━━━━━━━━━━━ 42s 102ms/step - categorical_accuracy: 0.9426 - loss: 0.1877 - val_categorical_accuracy: 0.9901 - val_loss: 0.5189
Epoch 2/10
174/177 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - categorical_accuracy: 0.9881 - loss: 0.0542
Epoch 2: loss improved from 0.11005 to 0.06710, saving model to saved_models/model1.keras
177/177 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - categorical_accuracy: 0.9880 - loss: 0.0545 - val_categorical_accuracy: 0.9901 - val_loss: 0.3443
Epoch 3/10
173/177 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - categorical_accuracy: 0.9892 - loss: 0.0492
Epoch 3: loss improved from 0.06710 to 0.04310, saving model to saved_models/model1.keras
177/177 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - categorical_accuracy: 0.9892 - loss: 0.0490 - val_categorical_accuracy: 0.9730 - val_loss: 0.1460
Epoch 4/1

In [36]:
from tensorflow.keras.callbacks import ModelCheckpoint
import os

# Directory to save models
model_dir = "saved_models"
os.makedirs(model_dir, exist_ok=True)

# Load the trained models
top_words = len(ngram_dict)  # Size of the n-gram dictionary
o_dim = len(np.unique(y_train))  # Number of unique classes in y_train
seg_size = 200  # Sequence length

def load_model(model_name):
    model = DC_CNN_Model(top_words, seg_size, o_dim)
    model.load_weights(os.path.join(model_dir, f"{model_name}.keras"))
    return model

trained_models = {
    "model1": load_model("model1"),
    "model2": load_model("model2"),
    "model3": load_model("model3"),
    "model4": load_model("model4")
}

# Test models
test_results = {}
for model_name in trained_models:
    padded_x_test = pad_sequences(x_test_ngram, maxlen=seg_size, padding='post')
    padded_y_test = pad_sequences(y_test_ngram, maxlen=seg_size, padding='post')
    test_results[model_name] = trained_models[model_name].evaluate(
        padded_x_test,
        padded_y_test,
        verbose=1
    )
    print(f"Test results for {model_name}: {test_results[model_name]}")

# Evaluate models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score

# Function to calculate metrics
def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    aupr = average_precision_score(y_true, y_pred)
    return accuracy, precision, recall, f1, aupr

for model_name in trained_models:
    y_pred = trained_models[model_name].predict(x_test_ngram)
    y_pred = (y_pred > 0.5).astype(int)
    metrics = calculate_metrics(y_test_ngram, y_pred)
    print(f"Metrics for {model_name}: Accuracy: {metrics[0]}, Precision: {metrics[1]}, Recall: {metrics[2]}, F1: {metrics[3]}, AUPR: {metrics[4]}")

# Prediction function
def predict_function(model, sequence):
    # Convert sequence to n-grams
    sequence_ngrams = ngram_encode([sequence], chunk_size, ngram_dict)
    # Pad sequence
    padded_sequence = pad_sequences(sequence_ngrams, maxlen=seg_size, padding='post')
    # Predict
    prediction = model.predict(padded_sequence)
    return "Function" if prediction > 0.5 else "Non-function"

# Example usage
best_model_name = max(test_results, key=lambda k: test_results[k][1])  # Assuming the second element is accuracy
best_model = trained_models[best_model_name]
user_sequence = "Your protein sequence here"
predicted_function = predict_function(best_model, user_sequence)
print(f"Predicted function: {predicted_function}")


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(32, 200), output.shape=(32, 2)

In [ ]:
def predict_protein_function(model_name, protein_sequence, chunk_size, ngram_dict, seq_len):
    model = load_model(model_name)
    
    # Convert the protein sequence to n-grams
    protein_ngram = ngram_encode([protein_sequence], chunk_size, ngram_dict)
    
    # Pad the sequence
    protein_padded = pad_data(protein_ngram, seq_len)
    
    # Predict the function
    prediction = model.predict(protein_padded)
    return prediction

# Example usage: Predict protein function
best_model_name = os.path.join(model_dir, "model4.h5")  # Replace with the best model based on performance
protein_sequence = [1, 2, 3, 4, 5, 6]  # Example sequence, replace with actual input
prediction = predict_protein_function(best_model_name, protein_sequence, chunk_size, ngram_dict, seg_size)
print(f"Predicted protein function: {prediction}")
